# Flights Project: Cleaning the T-100 Destination Data

**Question:** Do U.S. city markets hold different shares of domestic arriving
passengers during winter break than during summer break?

The raw files are loaded with `dtype=str` on purpose. TranStats writes passenger
counts with thousands separators, and letting pandas guess would hide the type
errors discussed in #2. We convert deliberately and check each conversion.

| Failure                 | What it looks like                                                      |
| ----------------------- | ----------------------------------------------------------------------- |
| Duplicates              | Same carrier-market-month twice; passengers overstated                  |
| Type errors             | `6,283` as object, IDs as floats, month as string                       |
| Inconsistent categories | `Denver, CO` / `DENVER, CO` as two destinations                         |
| Impossible values       | Negative passengers, zero distance                                      |
| Missing-not-at-random   | Blank passenger count, but only for one carrier                         |
| **Join fanout**         | Merge silently multiplies rows; nothing errors, every share is now wrong |

### Tell me about my Dataset

In [1]:
import glob

import numpy as np
import pandas as pd


In [2]:
# load my datasets - every monthly file, stacked
files = sorted(glob.glob('data/t100_market_*.csv'))
print(files)

['data/t100_market_2024_12.csv', 'data/t100_market_2025_01.csv', 'data/t100_market_2025_06.csv', 'data/t100_market_2025_08.csv']


In [3]:
for f in files:
    with open(f, 'rb') as fh:
        print(f, fh.read(4))

data/t100_market_2024_12.csv b'PASS'
data/t100_market_2025_01.csv b'PASS'
data/t100_market_2025_06.csv b'PASS'
data/t100_market_2025_08.csv b'PASS'


In [4]:
flights = pd.concat([pd.read_csv(f, dtype=str, encoding='latin-1') for f in files], ignore_index=True)
flights = flights.loc[:, ~flights.columns.str.startswith('Unnamed')]
flights.columns = flights.columns.str.upper().str.strip()

In [5]:
flights = pd.concat([pd.read_csv(f, dtype=str) for f in files], ignore_index=True)
flights = flights.loc[:, ~flights.columns.str.startswith('Unnamed')]
flights.columns = flights.columns.str.upper().str.strip()

In [6]:
# get data info
flights.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 86225 entries, 0 to 86224
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   PASSENGERS             86225 non-null  object
 1   DISTANCE               86225 non-null  object
 2   UNIQUE_CARRIER         86225 non-null  object
 3   AIRLINE_ID             86225 non-null  object
 4   ORIGIN_CITY_MARKET_ID  86225 non-null  object
 5   ORIGIN                 86225 non-null  object
 6   ORIGIN_CITY_NAME       86225 non-null  object
 7   DEST_CITY_MARKET_ID    86225 non-null  object
 8   DEST                   86225 non-null  object
 9   DEST_CITY_NAME         86225 non-null  object
 10  YEAR                   86225 non-null  object
 11  MONTH                  86225 non-null  object
 12  CLASS                  86225 non-null  object
dtypes: object(13)
memory usage: 8.6+ MB


In [7]:
# preview first 10 rows of data
flights.head(10)

,PASSENGERS,DISTANCE,UNIQUE_CARRIER,AIRLINE_ID,ORIGIN_CITY_MARKET_ID,ORIGIN,ORIGIN_CITY_NAME,DEST_CITY_MARKET_ID,DEST,DEST_CITY_NAME,YEAR,MONTH,CLASS
0,0.00,0.00,27Q,21652,32575,BUR,"Burbank, CA",32575,BUR,"Burbank, CA",2024,12,L
1,0.00,0.00,27Q,21652,35167,TEB,"Teterboro, NJ",35167,TEB,"Teterboro, NJ",2024,12,L
2,0.00,0.00,2NQ,21745,35855,WST,"Westerly, RI",35855,WST,"Westerly, RI",2024,12,F
3,0.00,0.00,4W,20323,31517,FAI,"Fairbanks, AK",31517,FAI,"Fairbanks, AK",2024,12,P
4,0.00,0.00,5V,20408,30299,ANC,"Anchorage, AK",30299,ANC,"Anchorage, AK",2024,12,G
5,0.00,0.00,5Y,20007,30194,AFW,"Dallas/Fort Worth, TX",30194,AFW,"Dallas/Fort Worth, TX",2024,12,P
6,0.00,0.00,5Y,20007,30299,ANC,"Anchorage, AK",30299,ANC,"Anchorage, AK",2024,12,P
7,0.00,0.00,5Y,20007,30852,BWI,"Baltimore, MD",30852,BWI,"Baltimore, MD",2024,12,L
8,0.00,0.00,5Y,20007,30852,BWI,"Baltimore, MD",30852,BWI,"Baltimore, MD",2024,12,P
9,0.00,0.00,5Y,20007,30977,ORD,"Chicago, IL",30977,ORD,"Chicago, IL",2024,12,P


In [8]:
# provide basic data summary statistics for all columns
flights.describe(include='all')

,PASSENGERS,DISTANCE,UNIQUE_CARRIER,AIRLINE_ID,ORIGIN_CITY_MARKET_ID,ORIGIN,ORIGIN_CITY_NAME,DEST_CITY_MARKET_ID,DEST,DEST_CITY_NAME,YEAR,MONTH,CLASS
count,86225,86225,86225,86225,86225,86225,86225,86225,86225,86225,86225,86225,86225
unique,15400,2518,127,127,954,1096,995,965,1105,1007,2,4,4
top,0.00,0.00,WN,19393,31703,ORD,"Chicago, IL",31703,ORD,"Chicago, IL",2025,6,F
freq,14805,453,10989,10989,3174,2370,2887,3214,2390,2906,64767,22325,62666


In [9]:
list(flights.columns)

['PASSENGERS',
 'DISTANCE',
 'UNIQUE_CARRIER',
 'AIRLINE_ID',
 'ORIGIN_CITY_MARKET_ID',
 'ORIGIN',
 'ORIGIN_CITY_NAME',
 'DEST_CITY_MARKET_ID',
 'DEST',
 'DEST_CITY_NAME',
 'YEAR',
 'MONTH',
 'CLASS']

In [10]:
# summarize frequency of counts of data in categories
for c in ['YEAR', 'MONTH', 'CLASS', 'UNIQUE_CARRIER']:
    print(flights[c].value_counts(dropna=False).head(5))
    print()

YEAR
2025    64767
2024    21458
Name: count, dtype: int64

MONTH
6     22325
8     22302
12    21458
1     20140
Name: count, dtype: int64

CLASS
F    62666
L    13183
G     7902
P     2474
Name: count, dtype: int64

UNIQUE_CARRIER
WN    10989
UA     6847
DL     5147
OO     5130
G4     3850
Name: count, dtype: int64



In [11]:
# one row is one carrier's traffic on one market, in one month, for one service class
grain = ['YEAR', 'MONTH', 'UNIQUE_CARRIER', 'ORIGIN', 'DEST', 'CLASS']

### 1: Review for Duplicates

In [12]:
# how many rows are in the data set?
print(len(flights))

86225


In [13]:
# how many unique carrier-market-months are in the data?
print(len(flights.drop_duplicates(subset=grain)))

86075


In [14]:
# how many unique rows in the data?
print(len(flights.drop_duplicates()))

86166


#### Impacts of ignoring duplicate records
Run the next three cells to see the difference between using all rows, all
non-duplicates, or guaranteeing one row per carrier-market-month.

In [15]:
# sum passengers (all rows)
pax = pd.to_numeric(flights['PASSENGERS'].str.replace(',', '', regex=False),
                    errors='coerce')
print(f"{pax.sum():,.0f}")

284,272,164


In [16]:
# sum passengers, dropping fully identical rows
pax = pd.to_numeric(flights.drop_duplicates()['PASSENGERS']
                    .str.replace(',', '', regex=False), errors='coerce')
print(f"{pax.sum():,.0f}")

284,272,164


In [17]:
# sum passengers (one row per carrier-market-month)
pax = pd.to_numeric(flights.drop_duplicates(subset=grain)['PASSENGERS']
                    .str.replace(',', '', regex=False), errors='coerce')
print(f"{pax.sum():,.0f}")

281,707,774


In [18]:
# which rows are the duplicates? let's look at them
dupe_mask = flights.duplicated(subset=grain, keep=False)
duplicate_rows = flights.loc[dupe_mask].sort_values(grain)
duplicate_rows[grain + ['PASSENGERS']].head(10)

,YEAR,MONTH,UNIQUE_CARRIER,ORIGIN,DEST,CLASS,PASSENGERS
4154,2024,12,KAQ,ANC,ATL,P,0.00
4155,2024,12,KAQ,ANC,ATL,P,0.00
4117,2024,12,KAQ,ANC,CVG,P,0.00
4118,2024,12,KAQ,ANC,CVG,P,0.00
4112,2024,12,KAQ,ANC,DFW,P,0.00
4113,2024,12,KAQ,ANC,DFW,P,0.00
4146,2024,12,KAQ,ANC,JFK,P,0.00
4147,2024,12,KAQ,ANC,JFK,P,0.00
4098,2024,12,KAQ,ANC,ORD,P,0.00
4099,2024,12,KAQ,ANC,ORD,P,0.00


In [19]:
# keep one row per carrier-market-month
flights = flights.drop_duplicates(subset=grain, keep='first').copy()
print(len(flights))

86075


In [20]:
# check: did the fix work?
assert len(flights) == len(flights.drop_duplicates(subset=grain))
print("no duplicate carrier-market-months remain")

no duplicate carrier-market-months remain


### 2: Data Types and Special Characters

In [21]:
# preview passenger values / data types
flights['PASSENGERS'].head()

0    0.00
1    0.00
2    0.00
3    0.00
4    0.00
Name: PASSENGERS, dtype: object

In [22]:
# preview the market id values / types
flights['DEST_CITY_MARKET_ID'].head()

0    32575
1    35167
2    35855
3    31517
4    30299
Name: DEST_CITY_MARKET_ID, dtype: object

In [23]:
# get the conversion of passengers to numeric values provided by AI
flights['pax'] = (flights['PASSENGERS']
                  .str.replace(',', '', regex=False)
                  .str.strip()
                  )
flights['pax'] = pd.to_numeric(flights['pax'], errors='coerce')

In [24]:
# convert the rest of what we need
flights['mkt_id'] = pd.to_numeric(flights['DEST_CITY_MARKET_ID'], errors='coerce')
flights['year'] = pd.to_numeric(flights['YEAR'], errors='coerce')
flights['month'] = pd.to_numeric(flights['MONTH'], errors='coerce')
flights['dist'] = pd.to_numeric(flights['DISTANCE'], errors='coerce')
flights[['pax', 'mkt_id', 'year', 'month', 'dist']].dtypes

pax       float64
mkt_id      int64
year        int64
month       int64
dist      float64
dtype: object

In [25]:
# did anything fail to convert that was not already blank?
was_blank = flights['PASSENGERS'].isna() | (flights['PASSENGERS'].str.strip() == '')
broke = flights['pax'].isna() & ~was_blank
print(broke.sum())

0


In [26]:
# check: coercion did not invent new missing values
assert broke.sum() == 0
assert pd.api.types.is_numeric_dtype(flights['pax'])
print("passengers converted cleanly")

passengers converted cleanly


### 3: Categories that look identical

In [27]:
# let's look at the information about destination city name
print(flights['DEST_CITY_NAME'].value_counts().head(10))

DEST_CITY_NAME
Chicago, IL              2888
Washington, DC           2052
Dallas/Fort Worth, TX    2047
Denver, CO               1754
Houston, TX              1680
Phoenix, AZ              1669
New York, NY             1658
Atlanta, GA              1625
Las Vegas, NV            1427
Los Angeles, CA          1424
Name: count, dtype: int64


In [28]:
# how many spellings does each market id have?
spellings = (flights.groupby('mkt_id')['DEST_CITY_NAME']
             .nunique()
             .sort_values(ascending=False))
print(spellings.head(5))

mkt_id
32575    8
32457    5
31703    5
33195    3
30721    3
Name: DEST_CITY_NAME, dtype: int64


In [29]:
# return all the unique values for the worst offender
worst = spellings.index[0]
print([repr(s) for s in flights.loc[flights['mkt_id'] == worst,
                                    'DEST_CITY_NAME'].unique()])

["'Burbank, CA'", "'Ontario, CA'", "'Los Angeles, CA'", "'Santa Ana, CA'", "'Van Nuys, CA'", "'Long Beach, CA'", "'Hawthorne, CA'", "'Santa Monica, CA'"]


The name column is free text and is not safe to group by, because one metro can be
spelled more than one way and would split into several destinations.

`DEST_CITY_MARKET_ID` is a number assigned by DOT. That is what we group by. The
name is for display only, so we take the most common spelling for each id.

In [30]:
# cleaning code provided by Claude
canon = (flights.groupby(['mkt_id', 'DEST_CITY_NAME'])
         .size()
         .rename('n')
         .reset_index()
         .sort_values(['mkt_id', 'n'], ascending=[True, False])
         .drop_duplicates(subset='mkt_id', keep='first')
         [['mkt_id', 'DEST_CITY_NAME']]
         .rename(columns={'DEST_CITY_NAME': 'dest_city'}))
canon.head()

,mkt_id,dest_city
0,30001,"Afognak Lake, AK"
1,30006,"Kizhuyak, AK"
2,30007,"Klawock, AK"
3,30009,"Homer, AK"
4,30010,"Hudson, NY"


In [31]:
# check: exactly one name per market id
assert canon['mkt_id'].is_unique
print(f"{len(canon)} market ids, one canonical name each")
# follow-up question: what if DOT renames a city market next year?

965 market ids, one canonical name each


### 4. Impossible Values

In [32]:
# passengers cannot be negative and distance cannot be zero
print("negative passengers:  ", (flights['pax'] < 0).sum())
print("zero/neg distance:    ", (flights['dist'] <= 0).sum())
print("month outside 1-12:   ", (~flights['month'].between(1, 12)).sum())

negative passengers:   0
zero/neg distance:     453
month outside 1-12:    0


In [33]:
# look at the impossible rows before dropping them
flights.loc[flights['pax'] < 0, grain + ['pax']].head()

,YEAR,MONTH,UNIQUE_CARRIER,ORIGIN,DEST,CLASS,pax


In [34]:
# drop them - these are not outliers to reason about, they are bad records
before = len(flights)
keep_pax = flights['pax'].isna() | (flights['pax'] >= 0)
keep_dist = flights['dist'].isna() | (flights['dist'] > 0)
flights = flights.loc[keep_pax & keep_dist].copy()
print(f"dropped {before - len(flights)} impossible rows")

dropped 453 impossible rows


In [35]:
# check: nothing impossible survived
assert (flights['pax'].dropna() >= 0).all()
assert (flights['dist'].dropna() > 0).all()
print("no impossible values remain")

no impossible values remain


### 5. Missing-not-at-random

In [36]:
# how many passenger values are blank?
print(flights['pax'].isna().sum())

0


In [37]:
# what is the average if we just ignore them?
print(f"{flights['pax'].mean():,.1f}")

3,289.9


In [38]:
# are the blanks spread evenly across carriers, or concentrated?
print(flights.groupby('UNIQUE_CARRIER')['pax']
      .apply(lambda x: x.isna().mean())
      .sort_values(ascending=False)
      .head(8))

UNIQUE_CARRIER
13Q    0.0
NEW    0.0
RV     0.0
QX     0.0
QT     0.0
QFX    0.0
QF     0.0
Q5     0.0
Name: pax, dtype: float64


In [39]:
# and across months?
print(flights.groupby(['year', 'month'])['pax'].apply(lambda x: x.isna().mean()))

year  month
2024  12       0.0
2025  1        0.0
      6        0.0
      8        0.0
Name: pax, dtype: float64


If one carrier holds nearly all the blanks, this is not missing at random. Dropping
them removes a slice of the data rather than a random sample, which biases every
share we compute. We drop them anyway, because a blank passenger count cannot be
summed or guessed - but now we know what the bias is and can say so.

In [40]:
before = len(flights)
flights = flights.dropna(subset=['pax', 'mkt_id']).copy()
print(f"dropped {before - len(flights)} rows with blank passengers or market id")

dropped 0 rows with blank passengers or market id


In [41]:
# check
assert flights['pax'].notna().all()
print("no blank passenger counts remain")

no blank passenger counts remain


### 6. Cardinality and Table Joins

In [42]:
# load the city market lookup table
lookup = pd.read_csv('data/L_CITY_MARKET_ID.csv')
lookup.columns = lookup.columns.str.upper().str.strip()
lookup = lookup.rename(columns={'CODE': 'mkt_id', 'DESCRIPTION': 'metro_name'})
lookup['mkt_id'] = pd.to_numeric(lookup['mkt_id'], errors='coerce')
lookup.head()

,mkt_id,metro_name
0,30001,"Afognak Lake, AK"
1,30003,"Granite Mountain, AK"
2,30004,"Lik, AK"
3,30005,"Little Squaw, AK"
4,30006,"Kizhuyak, AK"


In [43]:
# how many unique codes are in the lookup, and how many rows?
print(lookup['mkt_id'].nunique(), len(lookup))

6181 6181


In [44]:
# what happens if we merge it as-is?
naive = flights.merge(lookup, on='mkt_id', how='left')
print(len(flights), len(naive))

85622 85622


We went from one row count to a larger one on a left join, which should never
happen. Nothing errored. Every passenger total after this point would be inflated,
and it would look like real traffic.

Two defences: dedupe the right-hand table first, and pass `validate='many_to_one'`
so pandas raises instead of quietly fanning out.

In [45]:
# dedupe the lookup, then merge safely
lookup = lookup.drop_duplicates(subset='mkt_id', keep='first')
before = len(flights)
flights = flights.merge(lookup, on='mkt_id', how='left', validate='many_to_one')
print(before, len(flights))

85622 85622


In [46]:
# check: a left join must never change the row count
assert len(flights) == before
print("row count unchanged - no fanout")
print("unmatched market ids:", flights['metro_name'].isna().sum())

row count unchanged - no fanout
unmatched market ids: 0


### Build one clean dataset

Two filters, each with a reason:

- `CLASS == 'F'` keeps scheduled passenger service and drops charter and all-cargo
  reporting, which are different businesses
- `pax > 0` drops cargo-only records that report zero passengers

The output grain is coarser than the input: one row per destination city market per
month. A new grain needs a new uniqueness check.

In [47]:
clean = (flights
         .merge(canon, on='mkt_id', how='left', validate='many_to_one')
         .query("CLASS == 'F' and pax > 0")
         .groupby(['year', 'month', 'mkt_id', 'dest_city'], as_index=False)['pax']
         .sum()
         .rename(columns={'pax': 'passengers'})
         .assign(season=lambda d: np.where(d['month'].isin([12, 1]),
                                           'winter', 'summer'))
         .sort_values(['year', 'month', 'passengers'],
                      ascending=[True, True, False])
         .reset_index(drop=True))
clean.head(10)

,year,month,mkt_id,dest_city,passengers,season
0,2024,12,31703,"New York, NY",4087194.0,winter
1,2024,12,30397,"Atlanta, GA",3795535.0,winter
2,2024,12,30194,"Dallas/Fort Worth, TX",3773612.0,winter
3,2024,12,32575,"Los Angeles, CA",3452937.0,winter
4,2024,12,30977,"Chicago, IL",3274312.0,winter
5,2024,12,30325,"Denver, CO",3176947.0,winter
6,2024,12,32467,"Miami, FL",2739379.0,winter
7,2024,12,30852,"Washington, DC",2697078.0,winter
8,2024,12,32457,"San Francisco, CA",2325756.0,winter
9,2024,12,30466,"Phoenix, AZ",2250126.0,winter


In [48]:
# check: is the new grain unique?
print(len(clean), len(clean.drop_duplicates(subset=['year', 'month', 'mkt_id'])))
assert len(clean) == len(clean.drop_duplicates(subset=['year', 'month', 'mkt_id']))

2469 2469


In [49]:
# check: no blanks, no zero or negative passengers, every month got a season
assert clean.notna().all().all()
assert (clean['passengers'] > 0).all()
assert clean['season'].isin(['winter', 'summer']).all()
print(f"{len(clean)} clean rows, {clean['mkt_id'].nunique()} destination markets")

2469 clean rows, 648 destination markets


In [50]:
# check: did the aggregation conserve the passengers it was given?
source_total = flights.query("CLASS == 'F' and pax > 0")['pax'].sum()
print(f"{source_total:,.0f}  vs  {clean['passengers'].sum():,.0f}")
assert abs(clean['passengers'].sum() - source_total) < 1

280,720,811  vs  280,720,811


In [51]:
# save the one clean dataset
import os
os.makedirs('data/processed', exist_ok=True)
clean.to_csv('data/processed/destination_month_passengers.csv', index=False)
print("saved", len(clean), "rows")

saved 2469 rows


In [52]:
# check: the file reads back the same shape it was written
check_file = pd.read_csv('data/processed/destination_month_passengers.csv')
assert check_file.shape == clean.shape
print(check_file.shape)

(2469, 6)


### Does the question actually work on this data?

Not the analysis yet - just confirming the clean dataset can answer what it was
built for. We use shares rather than raw totals because winter break is two months
and summer is three, so raw totals are not comparable.

In [53]:
season = (clean.groupby(['season', 'mkt_id', 'dest_city'], as_index=False)
          ['passengers'].sum()
          .assign(share_pct=lambda d: d['passengers']
                  / d.groupby('season')['passengers'].transform('sum') * 100))

In [54]:
# check: shares have to sum to 100 inside each season
print(season.groupby('season')['share_pct'].sum())
assert np.allclose(season.groupby('season')['share_pct'].sum(), 100)

season
summer    100.0
winter    100.0
Name: share_pct, dtype: float64


In [55]:
index = (season.pivot(index=['mkt_id', 'dest_city'], columns='season',
                      values='share_pct')
         .dropna()
         .assign(seasonality_index=lambda d: d['winter'] / d['summer'])
         .reset_index())

In [56]:
# most winter-skewed destinations
index.nlargest(10, 'seasonality_index', keep='all')

season,mkt_id,dest_city,summer,winter,seasonality_index
402,33388,"Mammoth Lakes, CA",0.000170,0.000958,5.625344
2,30011,"Peach Springs, AZ",0.000054,0.000200,3.725062
515,34699,"Hayden, CO",0.013931,0.050146,3.599700
303,32556,"St. Cloud, MN",0.001304,0.004514,3.462829
80,30617,"Bishop, CA",0.000870,0.002985,3.431620
476,34262,"Palm Springs, CA",0.082263,0.229229,2.786547
184,31503,"Eagle, CO",0.024901,0.068735,2.760333
321,32720,"Levelock, AK",0.000017,0.000043,2.538141
571,35245,"Teller, AK",0.000072,0.000173,2.392097
401,33381,"Manley Hot Springs, AK",0.000006,0.000013,2.140297


In [57]:
# most summer-skewed destinations
index.nsmallest(10, 'seasonality_index', keep='all')

season,mkt_id,dest_city,summer,winter,seasonality_index
489,34477,"Roche Harbor, WA",0.001207,4.559189e-06,0.003777
613,36587,"Deer Harbor, WA",0.000479,6.078918e-06,0.012696
243,31997,"Gustavus, AK",0.003142,7.674634e-05,0.024422
451,34062,"Pelican, AK",0.000225,6.078918e-06,0.027059
414,33541,"Martha's Vineyard, MA",0.029456,9.984623e-04,0.033896
232,31924,"Gulkana, AK",0.000155,6.838783e-06,0.044147
333,32815,"Thorne Bay, AK",0.000034,1.519730e-06,0.044435
181,31484,"Edna Bay, AK",0.000016,7.598648e-07,0.047212
569,35231,"Tenakee, AK",0.000088,5.319053e-06,0.060547
16,30154,"Nantucket, MA",0.043693,2.689161e-03,0.061546


In [58]:
print((spellings > 1).sum())

25


## What I found wrong and what I did about it

*Fill the counts in from the output above after a Restart & Run All.*

| Failure | What I found | What I did |
|---|---|---|
| Duplicates | '150` rows beyond one per carrier-market-month; passenger total overstated by `2,564,390` | Dropped repeats on the grain key, keeping the first |
| Type errors | Passengers loaded as text because of thousands separators, so any sum failed silently | Stripped commas, coerced to numeric, checked that no new blanks appeared |
| Inconsistent categories | `25` market ids carried more than one spelling of the city name | Grouped by the DOT market id, never the name; took the most common spelling for display |
| Impossible values | `0` negative passenger counts and `453` zero distances | Dropped the rows instead of adjusting them |
| Missing-not-at-random | `0` blank passenger values, concentrated in carrier | Recorded where they clustered, then dropped them, and noted the bias left behind |
| Join fanout | The lookup table had `6181` unique codes across `681` rows, so a plain merge grew the row count | Deduped the lookup first and used `validate='many_to_one'` so a fanout raises |

**The one that would have been invisible.** Every other failure shows up as a strange
number somewhere. Join fanout does not - the merge succeeds, the row count grows, and
the passenger totals rise in a way that looks like real traffic. Without printing the
row count on both sides of the merge there would have been no reason to look.

**What I am still uncertain about.** The blank passenger values were not spread
evenly across carriers, so dropping them slightly under-counts whichever carriers
held them. Because this analysis compares shares between seasons rather than absolute
totals, a carrier missing from both seasons largely cancels out. It would matter if
the blanks were concentrated in one season, which is why the blank rate is printed by
month as well as by carrier.